## Preparing Data for a model to forecast the country's total energy consumption

As the title suggests, in this notebook, I intend to focus on preparing data for the model.

I am beginning the load forecasting task immediately after the successful implementation of an offshore wind power forecasting model, as I believe that features derived from the load data could be useful when developing more complex models later on.

In my current approach, I am operating on the premise that the load is, to a large extent, dependent on calendar-based features.

**However, the plan is to try incorporating features such as temperature or storm warnings into the set of predictors, as these can affect heating costs or, theoretically, lead to the suspension of operations and a drop in load.**

In [4]:
# Imports
import pandas as pd
import numpy as np
import os
import pyrootutils
import holidays
from dotenv import load_dotenv

# Needed to load variables from the .env file into the environment
load_dotenv()

# Determining the project root based on the .gitignore file
PROJECT_ROOT = pyrootutils.setup_root(
    search_from=__file__ if "__file__" in locals() else ".",
    indicator=".gitignore",
    pythonpath=True,
)
from utils.weather import fetch_load_weather

# Determining data directories
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw_data" / "load_model"
CLEAN_DATA_DIR = PROJECT_ROOT / "data" / "clean_data" / "load_model"
DATASETS_DIR = PROJECT_ROOT / "data" / "datasets" / "load_model"

### In the cell below, I'm creating the basic structure for the future dataset.
For now, I'll use `UTC time` as the index, as it's unique and will be useful for joins.

For the first column, I've used converted `Berlin time`, this will be used to calculate the temporal features for the models.

The step size is 15 minutes, as the target (`load_act`) has that resolution.

In [5]:
# Determining the time interval for the energy load data
START = '2026-01-01'
END = '2026-06-20'

# Creating a DataFrame with UTC and Berlin timestamps
# Index (UTC Timestamps)
utc_index = pd.date_range(
    start=START, 
    end=END, 
    freq='15min', 
    tz='UTC'
)

# Converting UTC timestamps to Berlin time
berlin_time = utc_index.tz_convert('Europe/Berlin')

# Combining both UTC and Berlin timestamps into a single DataFrame
df = pd.DataFrame({
    'dt_berlin': berlin_time
}, index=utc_index)

# Displaying a sample of the DataFrame to verify its structure
df.sample(5)  # Displaying a sample of the DataFrame to verify its structure

,dt_berlin
2026-05-13 15:45:00+00:00,2026-05-13 17:45:00+02:00
2026-03-31 18:45:00+00:00,2026-03-31 20:45:00+02:00
2026-04-16 03:15:00+00:00,2026-04-16 05:15:00+02:00
2026-05-17 18:15:00+00:00,2026-05-17 20:15:00+02:00
2026-03-10 02:15:00+00:00,2026-03-10 03:15:00+01:00


## Preparing Calendar Features
In this cell, I prepare calendar-based features. These include:
- `time_sin`, `time_cos` - local time **cyclically** encoded for continuity
- `quarter_index` - an **experimental** feature that shows the number of 15-minute intervals within a day, calculated from 0:00 **local** time, may be useful for boosting models
- `doy` - an **experimental** feature that shows the day of the year; may also be useful
- `doy_sin`, `doy_cos` - **continuously** encoded day of the year
- `holiday_index` - a **weighted** holiday index. To create the weight, the regions where the holiday takes place are multiplied by the annual total energy consumption of that region according to the Monitoringbericht 2025 report, after which it is normalized as a percentage from 0.0 to 1.0
- `workday_index` - an **experimental** flag indicating whether it is a workday, it excludes weekends and `holiday_index`
- `is_monday` ... `is_sunday` - **binary** flags indicating what day of the week it is

In [6]:
# Encoding the time of day using sine and cosine
local = df['dt_berlin']
minutes_from_midnight = local.dt.hour * 60 + local.dt.minute
time_phase = 2 * np.pi * minutes_from_midnight / 1440

df["time_sin"] = np.sin(time_phase)
df["time_cos"] = np.cos(time_phase)


# Creating a 15-minute day index
quarter_index = minutes_from_midnight // 15
df['quarter_index'] = quarter_index


# Adding a column for the day of the year based on Berlin time
df['doy'] = local.dt.dayofyear

# Encoding the day of the year using sine and cosine
# Calculating the number of days in the year for each date, accounting for leap years
days_in_year = 365 + df["dt_berlin"].dt.is_leap_year.astype(int)
doy_phase = 2 * np.pi * (df["doy"] - 1) / days_in_year

df["doy_sin"] = np.sin(doy_phase)
df["doy_cos"] = np.cos(doy_phase)

# Adding a column for the day of the week based on Berlin time
df['dow'] = local.dt.day_name()


# Adding a column for the holiday weigted index
# Information taken from:
# Monitoringbericht 2025 - Monitoringbericht gemäß § 63 Abs. 3 i. V. m. § 35 EnWG und § 48 Abs. 3 i. V. m. § 53 Abs. 3 GWB
# https://data.bundesnetzagentur.de/Bundesnetzagentur/SharedDocs/Mediathek/Monitoringberichte/MonitoringberichtEnergie2025.pdf
# Pages 96-97
# Consumption volumes (TWh) by states at the VNB level for 2024 (Bundesnetzagentur)
consumption_twh = {
    'NW': 86.94,  # Nordrhein-Westfalen
    'BY': 63.02,  # Bayern
    'BW': 52.09,  # Baden-Württemberg
    'HE': 40.45,  # Hessen
    'NI': 35.58,  # Niedersachsen
    'SN': 19.81,  # Sachsen
    'RP': 17.97,  # Rheinland-Pfalz
    'ST': 13.48,  # Sachsen-Anhalt
    'BE': 12.31,  # Berlin
    'BB': 11.59,  # Brandenburg
    'SH': 11.22,  # Schleswig-Holstein
    'TH': 10.05,  # Thüringen
    'HH': 9.91,   # Hamburg
    'MV': 6.06,   # Mecklenburg-Vorpommern
    'SL': 5.07,   # Saarland
    'HB': 4.47    # Bremen
}

# Normalizing the consumption volumes to get weights for each state
total_twh = sum(consumption_twh.values())
state_weights = {state: val / total_twh for state, val in consumption_twh.items()}


# Creating a column for the weighted holiday indicator based on the state weights and holidays
dates = pd.to_datetime(df['dt_berlin'].dt.date)
years = dates.dt.year.unique().tolist()

# Creating a dictionary of holiday calendars for each state in Germany for the relevant years
state_calendars = {
    state: holidays.Germany(subdiv=state, years=years)
    for state in state_weights
}

# Calculating the weight only for unique dates
unique_dates = dates.dt.date.unique()
date_to_weight = {
    d: sum(weight for state, weight in state_weights.items() if d in state_calendars[state])
    for d in unique_dates
}

# Mapping the calculated weights to the original DataFrame based on the date
df['holiday_index'] = dates.dt.date.map(date_to_weight)

# Creating a boolean mask to identify weekdays (Monday to Friday)
is_weekday = (local.dt.dayofweek < 5)

# Creating a new column 'workday_index' that combines the weekday and holiday information
df["workday_index"] = (
    is_weekday.astype(float)
    * (1.0 - df["holiday_index"])
)

days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

# Encoding the day of the week into one-hot encoded columns
encoded = pd.get_dummies(
    pd.Categorical(df["dow"], categories=days),
    dtype="int8"
).set_axis([f"is_{day.lower()}" for day in days], axis=1)

df = pd.concat([df.drop(columns="dow"), encoded.set_axis(df.index)], axis=1)

# Displaying a sample of the DataFrame to verify its structure
df.sample(5)

# "quarter_index", "doy", "workday_index"
# These signs are still candidates, they may contain valuable information for boosting, but this requires verification

,dt_berlin,time_sin,time_cos,quarter_index,doy,doy_sin,doy_cos,holiday_index,workday_index,is_monday,is_tuesday,is_wednesday,is_thursday,is_friday,is_saturday,is_sunday
2026-05-03 07:45:00+00:00,2026-05-03 09:45:00+02:00,0.555570,-0.831470,39,123,0.863142,-0.504961,0.0,0.0,0,0,0,0,0,0,1
2026-06-16 00:00:00+00:00,2026-06-16 02:00:00+02:00,0.500000,0.866025,8,167,0.280231,-0.959933,0.0,1.0,0,1,0,0,0,0,0
2026-03-27 12:30:00+00:00,2026-03-27 13:30:00+01:00,-0.382683,-0.923880,54,86,0.994218,0.107381,0.0,1.0,0,0,0,0,1,0,0
2026-05-30 21:30:00+00:00,2026-05-30 23:30:00+02:00,-0.130526,0.991445,94,150,0.545240,-0.838280,0.0,0.0,0,0,0,0,0,1,0
2026-02-22 14:00:00+00:00,2026-02-22 15:00:00+01:00,-0.707107,-0.707107,60,53,0.780296,0.625411,0.0,0.0,0,0,0,0,0,0,1


### Weather from individual forecast runs

The load profile uses land cells: temperature at 2 m (°C), wind speed at 10 m (m/s), shortwave radiation (W/m², mean over the preceding hour), and cloud cover (%). The grid targets about 20 locations; it is spatially uniform, not consumption-weighted. Freeze the coordinates alongside the raw data before comparing models.

For each Berlin delivery day, use the previous day's 00:00 UTC run. The fetcher retains one buffer hour on each side, including DST days. Keep `datetime_run_utc` when joining or interpolating: adjacent runs deliberately overlap in their buffer hours. The returned data is hourly and has not yet been joined to the quarter-hour load grid.

## Preparing geographic coordinates
I need to prepare N equidistant points covering the entire area of ​​Germany to collect weather data for them. For this, I'll use my utility.

In [7]:
from utils.germany_grid_map import generate_germany_grid, visualize_grid

# Use the repository's boundary snapshot for reproducible coordinates.
load_points = generate_germany_grid(
    n_points=20,
    states_url=(PROJECT_ROOT / 'utils/germany_states_low.geojson').as_uri(),
)
load_points

# Creates an .html file with a map and point locations
# visualize_grid(load_points)

[germany_grid] Loading federal state boundaries...
[germany_grid] Finding grid spacing for about 20 points...
[germany_grid] Grid spacing: 140.27 km
[germany_grid] Points inside Germany: 20
[germany_grid] Assigning points to federal states...
[germany_grid] Done. Total points: 20


{1: [8.935644, 48.358116, 'Baden-Württemberg'],
 2: [10.82771, 48.360139, 'Bayern'],
 3: [12.718808, 48.329824, 'Bayern'],
 4: [7.945168, 49.43719, 'Rheinland-Pfalz'],
 5: [9.879022, 49.455853, 'Baden-Württemberg'],
 6: [11.813026, 49.44134, 'Bayern'],
 7: [6.90947, 50.506619, 'Rheinland-Pfalz'],
 8: [8.886832, 50.542796, 'Hessen'],
 9: [10.865672, 50.544926, 'Thüringen'],
 10: [12.843362, 50.513005, 'Sachsen'],
 11: [7.84762, 51.620384, 'Nordrhein-Westfalen'],
 12: [9.873275, 51.640069, 'Niedersachsen'],
 13: [11.899109, 51.62476, 'Sachsen-Anhalt'],
 14: [13.922253, 51.574493, 'Brandenburg'],
 15: [8.832033, 52.72618, 'Niedersachsen'],
 16: [10.90829, 52.728431, 'Sachsen-Anhalt'],
 17: [12.98317, 52.694702, 'Brandenburg'],
 18: [9.866803, 53.823428, 'Schleswig-Holstein'],
 19: [11.996039, 53.80722, 'Mecklenburg-Vorpommern'],
 20: [8.770126, 54.909071, 'Schleswig-Holstein']}

## Downloading Weather Data
In this cell, I download weather data using my own utility. It loads the forecast single run after midnight UTC on day D-1 for delivery day D.
The resulting dataframe has the following structure:
- `datetime_run_utc` - the UTC date when the weather model run was performed, needed for navigation and debugging
- `datetime_utc` - the UTC date of the delivery period on day D, useful for navigation and joins
- `datetime_berlin` - the date converted to local time, initially needed for feature engineering, but already completed in the previous step, can be used for debugging
- `point_id` - the ID of the geographic point (probe) from which the weather data was taken, needed for feature engineering and weather feature aggregation, as well as for debugging
- `region` - the federal state in which the point is located, needed for debugging
- `latitude`, `longitude` - geographic latitude and longitude indicators, needed primarily for debugging
- `temperature_2m` - the outside temperature, **critical** information for Models
- `wind_speed_10m` - wind speed, can be very useful for accounting for the heating/cooling needs of buildings
- `shortwave_radiation` - sunlight, can be useful for accounting for the heating/cooling needs of buildings
- `cloud_cover` - an **experimental** feature indicating cloud cover, partially duplicated in `shortwave_radiation`, but may have additional predictive power

**Open Meteo Historical Forecast** provides 5,000 API requests per hour and 10,000 per day for free. One request equals one single run for one geographic point, the number of features does not affect the query weight.

In [8]:
# Define the time interval for fetching weather data
WEATHER_START = '2026-06-25'
WEATHER_END = '2026-11-17'

# WeatherSTART/WeatherEND below are interpreted as inclusive Berlin delivery dates for weather.
WEATHER_RUN_START = (pd.Timestamp(WEATHER_START) - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
WEATHER_RUN_END = (pd.Timestamp(WEATHER_END) - pd.DateOffset(days=1)).strftime('%Y-%m-%d')
FETCH_LOAD_WEATHER = False  # Enable explicitly for the historical download.

# 20 coordinates with a 3-second interval fit within the minute API-limits
if FETCH_LOAD_WEATHER:
    import importlib
    from utils import weather as weather_module

    weather_module = importlib.reload(weather_module)
    RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

    temp_checkpoint = RAW_DATA_DIR / '_tmp_load_ECMWF_singleruns.parquet'
    print(f'Weather module: {weather_module.__file__}')
    print(f'Temp checkpoint: {temp_checkpoint.resolve()}')

    load_weather = weather_module.fetch_load_weather(
        WEATHER_RUN_START,
        WEATHER_RUN_END,
        points=load_points,
        checkpoint_path=temp_checkpoint,
    )

    dt_utc = pd.to_datetime(load_weather['datetime_utc'], utc=True)
    actual_start = dt_utc.min().strftime('%Y-%m-%d')
    actual_end = dt_utc.max().strftime('%Y-%m-%d')

    stem = f'load_ECMWF_singleruns_{actual_start}_to_{actual_end}'
    final_parquet = RAW_DATA_DIR / f'{stem}.parquet'

    load_weather.to_parquet(final_parquet, index=False)
    if temp_checkpoint.exists() and temp_checkpoint != final_parquet:
        temp_checkpoint.unlink()

    pd.DataFrame.from_dict(
        load_points, orient='index', columns=['longitude', 'latitude', 'region']
    ).rename_axis('point_id').to_json(RAW_DATA_DIR / f'{stem}_points.json', orient='table')

    print(f'Saved parquet: {final_parquet.resolve()}')

In [13]:
# Checking the structure of the fetched weather data
if FETCH_LOAD_WEATHER:
    display(load_weather.sample(10))
else:
    display(pd.read_parquet( RAW_DATA_DIR / 'load_ECMWF_singleruns_2025-07-24_to_2026-06-23.parquet').sample(10))

,datetime_run_utc,datetime_utc,datetime_berlin,point_id,region,latitude,longitude,temperature_2m,wind_speed_10m,shortwave_radiation,cloud_cover
71832,2025-10-13 00:00:00+00:00,2025-10-14 17:00:00+00:00,2025-10-14 19:00:00+02:00,9,Thüringen,50.579964,10.831859,8.750000,2.128967,1.0,100.0
161451,2026-01-18 00:00:00+00:00,2026-01-19 14:00:00+00:00,2026-01-19 15:00:00+01:00,19,Mecklenburg-Vorpommern,53.813705,11.965318,2.700000,3.924602,167.0,0.0
136284,2026-02-24 00:00:00+00:00,2026-02-25 15:00:00+00:00,2026-02-25 16:00:00+01:00,16,Sachsen-Anhalt,52.688927,10.934580,12.550000,2.486966,211.0,55.0
121282,2026-05-28 00:00:00+00:00,2026-05-29 15:00:00+00:00,2026-05-29 17:00:00+02:00,14,Brandenburg,51.564144,13.883847,25.200001,2.304886,622.0,0.0
80352,2025-10-06 00:00:00+00:00,2025-10-07 09:00:00+00:00,2025-10-07 11:00:00+02:00,10,Sachsen,50.509663,12.879859,10.151000,3.373796,162.0,95.0
125883,2025-12-21 00:00:00+00:00,2025-12-22 14:00:00+00:00,2025-12-22 15:00:00+01:00,15,Niedersachsen,52.759224,8.764045,6.150000,5.800216,56.0,100.0
115265,2025-10-09 00:00:00+00:00,2025-10-10 04:00:00+00:00,2025-10-10 06:00:00+02:00,14,Brandenburg,51.564144,13.883847,11.750000,4.802343,0.0,100.0
162422,2026-02-24 00:00:00+00:00,2026-02-25 23:00:00+00:00,2026-02-26 00:00:00+01:00,19,Mecklenburg-Vorpommern,53.813705,11.965318,3.050000,4.583939,0.0,0.0
19274,2025-10-02 00:00:00+00:00,2025-10-03 05:00:00+00:00,2025-10-03 07:00:00+02:00,3,Bayern,48.330402,12.663317,2.500000,2.807134,0.0,0.0
105050,2025-08-12 00:00:00+00:00,2025-08-13 07:00:00+00:00,2025-08-13 09:00:00+02:00,13,Sachsen-Anhalt,51.634445,11.945454,21.150000,1.741408,323.0,0.0


## Checking the accuracy of geographic points.
Checking the accuracy of geographic points.
After loading, I noticed that the `latitude` and `longitude` parameters were slightly different from those requested.

The error is **small** and **shouldn't affect** the result for my task.

The point is that after transmitting the requested coordinates, Open Meteo converts them to its own grid. Therefore, just in case, I decided to check whether the coordinates were different between data collection sessions.

In [ ]:
import json

files = [
    RAW_DATA_DIR / "load_ECMWF_singleruns_2026-06-24_to_2026-11-17_points.json",
    RAW_DATA_DIR / "load_ECMWF_singleruns_2024-11-17_to_2025-07-25_points.json",
    RAW_DATA_DIR / "load_ECMWF_singleruns_2024-03-14_to_2024-11-19_points.json",
]

loaded = [json.loads(p.read_text(encoding="utf-8")) for p in files]

print(
    "1 and 2 are identical:", loaded[0] == loaded[1]
)
print(
    "1 and 3 are identical:", loaded[0] == loaded[2]
)
print(
    "2 and 3 are identical:", loaded[1] == loaded[2]
)

1 and 2 are identical: True
1 and 3 are identical: True
2 and 3 are identical: True
